# Phase 3 — Reply drafting + escalation decision

Input: `Data/amazon_conversation.csv` (5,000 conversations, intent-classified in Phase 2) and `Data/amazonhelp_threads.csv` (full conversation text). Output: `Data/agent_responses.csv` — one row per conversation with a draft reply (where possible), an escalate flag, and a stated reason.

**No resolution/outcome label exists in this dataset** (see `dataprep.ipynb` limitations). The proxy used here: **AmazonHelp's first reply** to a similar past customer message stands in for "how the brand resolved this." That's a real limitation, not a detail — there's no ground truth that the historical reply actually satisfied the customer, only that it's what AmazonHelp said.

**Retrieval**: for a new customer message, find the most similar past messages *with the same confidently-classified intent* (`intent_id` not null from Phase 2), using embedding cosine similarity. Restricting to the same intent keeps retrieval from grounding a reply in an unrelated past problem just because the wording happens to be similar; requiring `intent_id` not null keeps a bad Phase-2 classification from leaking into the grounding pool.

**Escalation is a rule, not another LLM call** — cheap, deterministic, and each decision comes with a reason a human can audit or dispute:

| condition | decision | reason |
|---|---|---|
| `intent_id` is null | escalate | Phase 2 wasn't confident enough to classify this at all |
| `predicted_intent == 'other'` | escalate | no defined resolution path for a catch-all intent |
| `predicted_intent` in `{account_access, billing_or_payment}` | escalate | high-stakes (security/money) — always human, regardless of confidence |
| no historical examples of this intent, or best similarity below threshold | escalate | grounding would be a guess, not a match |
| none of the above | auto-handle | confident intent, not high-stakes, strong historical match |

A draft reply is still generated for escalated rows whenever grounding examples exist (skipped only for null-intent and `other`) — a human reviewing an escalated ticket still benefits from a starting draft; it just isn't auto-sent.

In [1]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not found - check .env"

DATA = (Path.cwd() if (Path.cwd() / "Data").exists() else Path.cwd().parent) / "Data"
BRAND = "AmazonHelp"
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"
TOP_K = 3
MIN_SIMILARITY = 0.35
HIGH_STAKES_INTENTS = {"account_access", "billing_or_payment"}
DRAFT_BATCH_SIZE = 15
MAX_WORKERS = 4  # was 8 - hit org TPM rate limits (200k/min) on gpt-4o-mini at that concurrency

client = OpenAI()

In [2]:
conv = pd.read_csv(DATA / "amazon_conversation.csv")
threads = pd.read_csv(DATA / "amazonhelp_threads.csv", dtype={"in_response_to_tweet_id": "Int64"})

# The historical-resolution proxy: AmazonHelp's FIRST reply in each conversation (not the last) -
# we're drafting a first reply to a new message, so grounding on a brand's own first reply to a
# similar past message is the structurally matching comparison. Every kept conversation has at
# least one AmazonHelp turn (dataprep.ipynb assumption 2), so this is always defined.
brand_reply = (threads[threads["author_id"] == BRAND]
               .sort_values(["conv_id", "turn_index"])
               .groupby("conv_id").first()["text"]
               .rename("brand_reply"))

conv = conv.merge(brand_reply, on="conv_id", how="left")
assert conv["brand_reply"].notna().all(), "every conversation should have a brand reply"

mapped = conv[conv["intent_id"].notna()].reset_index(drop=True)
print(f"{len(conv):,} conversations total | {len(mapped):,} with a confident intent (the retrieval pool)")

5,000 conversations total | 3,143 with a confident intent (the retrieval pool)


In [3]:
def embed_all(texts, batch_size=100):
    vectors = []
    for start in range(0, len(texts), batch_size):
        chunk = texts[start:start + batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=chunk)
        vectors.extend(d.embedding for d in resp.data)
        print(f"  embedded {min(start + batch_size, len(texts))}/{len(texts)}", end="\r")
    print()
    return np.array(vectors)

embeddings = embed_all(mapped["text"].tolist())
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1.0
emb_norm = embeddings / norms
print(f"embeddings shape: {emb_norm.shape}")

  embedded 3143/3143


embeddings shape: (3143, 1536)


In [4]:
# Retrieval within each intent bucket only. For each row, find the TOP_K most similar OTHER
# conversations sharing the same intent_id, via cosine similarity on the embeddings above.
retrieval = {}  # conv_id -> list of {"similarity", "customer_text", "brand_reply"}, best first

for intent_id, g in mapped.groupby("intent_id"):
    idx = g.index.to_numpy()
    vecs = emb_norm[idx]
    sim = vecs @ vecs.T
    np.fill_diagonal(sim, -1.0)  # exclude self

    k = min(TOP_K, len(idx) - 1)
    conv_ids = g["conv_id"].to_numpy()
    texts = g["text"].to_numpy()
    replies = g["brand_reply"].to_numpy()

    if k <= 0:
        for cid in conv_ids:
            retrieval[cid] = []
        continue

    top = np.argpartition(-sim, kth=k - 1, axis=1)[:, :k]
    for row_pos, cid in enumerate(conv_ids):
        cand = top[row_pos]
        cand_sims = sim[row_pos, cand]
        order = np.argsort(-cand_sims)
        cand, cand_sims = cand[order], cand_sims[order]
        retrieval[cid] = [
            {"similarity": float(s), "customer_text": str(texts[p]), "brand_reply": str(replies[p])}
            for p, s in zip(cand, cand_sims) if s > -1.0
        ]

print(f"built retrieval sets for {len(retrieval):,} conversations")
example_cid = mapped.loc[mapped["intent_id"] == mapped["intent_id"].mode()[0], "conv_id"].iloc[0]
print(f"\nexample retrieval for conv_id={example_cid}:")
print(" query:", mapped.loc[mapped["conv_id"] == example_cid, "text"].iloc[0][:80])
for ex in retrieval[example_cid]:
    print(f"  sim={ex['similarity']:.2f} | {ex['customer_text'][:70]}")

built retrieval sets for 3,143 conversations

example retrieval for conv_id=2:
 query: Too many times now, my prime orders have been delayed, @115850! Your reliability
  sim=0.71 | Anyone else having issues with @115821 missing delivery times lately? 
  sim=0.71 | Hey @AmazonHelp pls stop using @115817 for Prime shipping until they g
  sim=0.70 | This is the second time my @115821 order has been late on delivery &am


In [5]:
def decide_escalation(row, examples):
    """Rule-based auto-handle/escalate decision with a stated reason. Returns (escalate, reason)."""
    if pd.isna(row["intent_id"]):
        return True, "intent unclear: Phase 2 classification confidence was below threshold"
    if row["predicted_intent"] == "other":
        return True, "intent classified as 'other' - no defined resolution path"
    if row["predicted_intent"] in HIGH_STAKES_INTENTS:
        return True, f"high-stakes intent ('{row['predicted_intent']}') always routed to a human"
    if not examples:
        return True, "no historical examples of this intent available to ground a reply"
    if examples[0]["similarity"] < MIN_SIMILARITY:
        return True, (f"best historical match too weak (similarity {examples[0]['similarity']:.2f} "
                       f"< {MIN_SIMILARITY})")
    return False, "confident intent, not high-stakes, strong historical match found"

conv["escalate"] = False
conv["escalate_reason"] = None
conv["top_similarity"] = np.nan
conv["n_grounding_examples"] = 0

for i, row in conv.iterrows():
    examples = retrieval.get(row["conv_id"], [])
    escalate, reason = decide_escalation(row, examples)
    conv.at[i, "escalate"] = escalate
    conv.at[i, "escalate_reason"] = reason
    conv.at[i, "n_grounding_examples"] = len(examples)
    if examples:
        conv.at[i, "top_similarity"] = examples[0]["similarity"]

# .at[] assignment of Python bools can leave the column as object dtype, on which pandas' unary
# `~` silently falls back to Python's bitwise invert (~True == -2, ~False == -1) instead of logical
# negation - cast explicitly so downstream boolean math (~conv["escalate"]) is actually correct.
conv["escalate"] = conv["escalate"].astype(bool)

# A draft is attempted whenever grounding exists, even for escalated rows (a human reviewing an
# escalated ticket still benefits from a starting draft) - skipped only when there's no intent at
# all or the intent is the undefined 'other' bucket.
conv["attempt_draft"] = (
    conv["intent_id"].notna() & (conv["predicted_intent"] != "other") & (conv["n_grounding_examples"] > 0)
)

print(conv["escalate"].value_counts().to_string())
print(f"\nattempting a draft for {conv['attempt_draft'].sum():,} / {len(conv):,} conversations")
print("\nescalation reasons:")
print(conv[conv["escalate"]]["escalate_reason"].value_counts().to_string())

escalate
False    2791
True     2209

attempting a draft for 3,132 / 5,000 conversations

escalation reasons:
escalate_reason
intent unclear: Phase 2 classification confidence was below threshold    1857
high-stakes intent ('billing_or_payment') always routed to a human        210
high-stakes intent ('account_access') always routed to a human            130
intent classified as 'other' - no defined resolution path                  11
best historical match too weak (similarity 0.33 < 0.35)                     1


In [6]:
DRAFT_SYSTEM_PROMPT = """You are drafting @AmazonHelp's reply to a customer tweet, in AmazonHelp's \
actual voice and format: brief, direct, helpful - usually 1-2 sentences, sometimes asking the customer \
to move to DM for order-specific details.

For each item you will see the new customer message and up to 3 similar past cases with the same \
intent, each showing a past customer's message and how AmazonHelp actually replied. Use those past \
replies as a style and resolution-approach guide - draft a NEW reply for the new message that follows \
the same kind of resolution approach, adapted to its specific details. Do not copy order numbers, \
names, or other specifics from the past cases into the new draft.

You will receive a JSON object {"items": [{"id": <int>, "customer_message": <string>, \
"similar_past_cases": [{"customer_message": <string>, "brand_reply": <string>}, ...]}, ...]}.

Reply with ONLY a JSON object {"results": [{"id": <int>, "draft_reply": <string>}, ...]}, one result \
per input item, same ids."""

def draft_batch(rows, retries=4):
    """rows: list of (conv_id, customer_text, examples). Returns {conv_id: draft_reply_or_None}."""
    payload = json.dumps({"items": [
        {"id": cid, "customer_message": text,
         "similar_past_cases": [{"customer_message": e["customer_text"], "brand_reply": e["brand_reply"]}
                                 for e in examples]}
        for cid, text, examples in rows
    ]}, ensure_ascii=False)
    last_err = None
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=CHAT_MODEL,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": DRAFT_SYSTEM_PROMPT},
                    {"role": "user", "content": payload},
                ],
            )
            parsed = json.loads(resp.choices[0].message.content)
            out = {int(r["id"]): r["draft_reply"] for r in parsed["results"]}
            if set(out) != {cid for cid, _, _ in rows}:
                raise ValueError("response ids don't match input ids")
            return out
        except Exception as e:  # noqa: BLE001
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    print(f"  batch of {len(rows)} failed after {retries} retries: {last_err!r}")
    return {cid: None for cid, _, _ in rows}

def draft_all(df, batch_size=DRAFT_BATCH_SIZE, max_workers=MAX_WORKERS):
    rows = [(r.conv_id, r.text, retrieval.get(r.conv_id, [])) for r in df.itertuples()]
    batches = [rows[i:i + batch_size] for i in range(0, len(rows), batch_size)]
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(draft_batch, b) for b in batches]
        for done, fut in enumerate(as_completed(futures), start=1):
            results.update(fut.result())
            print(f"  batches done: {done}/{len(batches)}", end="\r")
    print()
    return results

In [7]:
to_draft = conv[conv["attempt_draft"]]
print(f"drafting replies for {len(to_draft):,} conversations...")
draft_results = draft_all(to_draft)

conv["draft_reply"] = conv["conv_id"].map(draft_results)

agent_responses = conv[[
    "conv_id", "text", "predicted_intent", "confidence", "intent_id",
    "n_grounding_examples", "top_similarity", "escalate", "escalate_reason", "draft_reply",
]]
agent_responses.to_csv(DATA / "agent_responses.csv", index=False)
print(f"wrote {len(agent_responses):,} rows -> agent_responses.csv")

drafting replies for 3,132 conversations...


  batch of 15 failed after 4 retries: ValueError("response ids don't match input ids")


  batches done: 209/209
wrote 5,000 rows -> agent_responses.csv


In [8]:
# Backfill any drafts that failed (mostly rate-limit related at higher concurrency) without
# re-spending on rows that already succeeded.
missing = conv[conv["attempt_draft"] & conv["draft_reply"].isna()]
if len(missing):
    print(f"retrying {len(missing):,} failed drafts at MAX_WORKERS={MAX_WORKERS}...")
    retry_results = draft_all(missing, batch_size=DRAFT_BATCH_SIZE, max_workers=MAX_WORKERS)
    conv["draft_reply"] = conv["draft_reply"].fillna(conv["conv_id"].map(retry_results))

    agent_responses = conv[[
        "conv_id", "text", "predicted_intent", "confidence", "intent_id",
        "n_grounding_examples", "top_similarity", "escalate", "escalate_reason", "draft_reply",
    ]]
    agent_responses.to_csv(DATA / "agent_responses.csv", index=False)
    print(f"re-saved agent_responses.csv; still missing: {(conv['attempt_draft'] & conv['draft_reply'].isna()).sum():,}")
else:
    print("no missing drafts to backfill")

retrying 15 failed drafts at MAX_WORKERS=4...


  batches done: 1/1
re-saved agent_responses.csv; still missing: 0


In [9]:
n_auto = (~conv["escalate"]).sum()
n_escalate = conv["escalate"].sum()
n_drafted = conv["draft_reply"].notna().sum()
n_draft_failed = (conv["attempt_draft"] & conv["draft_reply"].isna()).sum()

print(f"auto-handle : {n_auto:,} ({n_auto / len(conv):.1%})")
print(f"escalate    : {n_escalate:,} ({n_escalate / len(conv):.1%})")
print(f"drafts written (incl. for escalated rows kept as human starting point): {n_drafted:,}")
print(f"drafts attempted but failed after retries: {n_draft_failed:,}")

print("\n--- 3 auto-handle examples ---")
for _, r in conv[~conv["escalate"] & conv["draft_reply"].notna()].sample(3, random_state=0).iterrows():
    print(f"\n[{r['predicted_intent']}, sim={r['top_similarity']:.2f}] {r['text'][:90]}")
    print(f"  draft: {r['draft_reply']}")

print("\n--- 3 escalated examples (with a draft still attached for the human) ---")
esc_with_draft = conv[conv["escalate"] & conv["draft_reply"].notna()]
for _, r in esc_with_draft.sample(min(3, len(esc_with_draft)), random_state=0).iterrows():
    print(f"\n[{r['predicted_intent']}] reason: {r['escalate_reason']}")
    print(f"  message: {r['text'][:90]}")
    print(f"  draft:   {r['draft_reply']}")

auto-handle : 2,791 (55.8%)
escalate    : 2,209 (44.2%)
drafts written (incl. for escalated rows kept as human starting point): 3,132
drafts attempted but failed after retries: 0

--- 3 auto-handle examples ---

[delivery_problem, sim=0.71] Beyond annoyed that ALL3  items I ordered on @115821 prime are taking 110 years to get her
  draft: I understand your frustration! Prime shipping time refers to delivery after shipping, typically calculated in business days. Let us know if your order hasn't arrived within the expected timeframe! ^AJ

[delivery_problem, sim=0.70] What do I think of @115821 delivery service? It's bullshit. My package says delivered but 
  draft: I'm sorry to hear about your delivery issue. Please check this link for steps: https://t.co/xCMI7SiuC0, and let us know what you find! ^EZ

[delivery_problem, sim=0.75] @AmazonHelp say they put my parcel through my letterbox on saturday. Still waiting for it 
  draft: Sorry to hear that your parcel hasn't come through yet! Ple

## Caveats

- **The "historical resolution" is AmazonHelp's first reply, not a verified successful outcome.** The dataset has no ticket status or customer-satisfaction signal, so a bad or unhelpful historical reply can still get retrieved and imitated if it's the most textually similar one available. This is the single biggest gap between what this pipeline does and what "grounded in how the brand has historically resolved similar issues" ideally means.
- **The retrieval pool is only the 3,143 confidently-classified conversations among the first 5,000** (chronologically oldest) — small and non-representative. Expanding Phase 2 to more/all conversations would directly improve grounding quality here.
- **`MIN_SIMILARITY = 0.35` and `TOP_K = 3` are unvalidated guesses**, exactly like the 75 confidence threshold in Phase 2. They should be tuned against the golden eval set once it exists, not treated as settled.
- **No automated check that a draft is actually good** — this notebook only reports whether a draft was produced, not whether it's accurate, on-brand, or safe to send. That's what the LLM-as-judge rubric (with human-agreement evidence) needs to cover next.
- **High-stakes intents escalate unconditionally**, which is a deliberate, conservative policy choice (favoring safety over automation rate) worth stating explicitly in the report rather than leaving implicit.